# Qwen3-14B SFT evaluation


In [ ]:
import os, subprocess, sys
from pathlib import Path

LAUNCH_DIR = Path.cwd().resolve()
subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv>=1,<2"], check=True)
from dotenv import load_dotenv

ENV_FILE = Path(os.environ.get("CRASHDIAG_ENV_FILE", LAUNCH_DIR / "env.txt")).expanduser()
if not ENV_FILE.is_absolute():
    ENV_FILE = (LAUNCH_DIR / ENV_FILE).resolve()
if ENV_FILE.is_file():
    load_dotenv(ENV_FILE, override=True)

try:
    from kaggle_secrets import UserSecretsClient
except ImportError:
    UserSecretsClient = None

KAGGLE_SECRET_ALIASES = {
    "HF_TOKEN": "HF_TOKEN",
    "CRASHDIAG_DATASET_RUN_ID": "CRASHDIAG_DATASET_RUN_ID",
    "DATASET_RUN_ID": "CRASHDIAG_DATASET_RUN_ID",
    "CRASHDIAG_SFT_RUN_ID": "CRASHDIAG_SFT_RUN_ID",
    "SFT_RUN_ID": "CRASHDIAG_SFT_RUN_ID",
    "CRASHDIAG_SFT_EVAL_RUN_ID": "CRASHDIAG_SFT_EVAL_RUN_ID",
    "CRASHDIAG_SANDBOX_URL": "CRASHDIAG_SANDBOX_URL",
    "CRASHDIAG_API_TOKEN": "CRASHDIAG_API_TOKEN",
    "CRASHDIAG_SANDBOX_TOKEN": "CRASHDIAG_API_TOKEN",
    "CRASHDIAG_SOURCE_COMMIT": "CRASHDIAG_SOURCE_COMMIT",
    "SOURCE_COMMIT": "CRASHDIAG_SOURCE_COMMIT",
}
loaded_kaggle_secrets = []
kaggle_secret_errors = {}
if UserSecretsClient is not None:
    secrets = UserSecretsClient()
    for secret_name, env_name in KAGGLE_SECRET_ALIASES.items():
        if os.environ.get(env_name):
            continue
        try:
            value = secrets.get_secret(secret_name)
        except Exception as exc:
            kaggle_secret_errors[secret_name] = f"{type(exc).__name__}: {exc}"
            continue
        if value:
            os.environ[env_name] = value
            loaded_kaggle_secrets.append(secret_name)
print("loaded Kaggle secret names:", loaded_kaggle_secrets or "none")
if not os.environ.get("CRASHDIAG_API_TOKEN") and os.environ.get("CRASHDIAG_SANDBOX_TOKEN"):
    os.environ["CRASHDIAG_API_TOKEN"] = os.environ["CRASHDIAG_SANDBOX_TOKEN"]
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

REPO_URL = os.environ.get("CRASHDIAG_REPO_URL", "https://github.com/Indium-AI-Labs/CrashDiag.git")
SOURCE_COMMIT = os.environ.get("CRASHDIAG_SOURCE_COMMIT", "main")
WORKDIR = Path(os.environ.get("CRASHDIAG_WORKDIR", LAUNCH_DIR / "CrashDiag-runtime")).expanduser().resolve()
if (WORKDIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
elif WORKDIR.exists() and any(WORKDIR.iterdir()):
    raise RuntimeError(f"CRASHDIAG_WORKDIR exists and is not a Git checkout: {WORKDIR}")
else:
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
subprocess.run(["git", "-C", str(WORKDIR), "checkout", SOURCE_COMMIT], check=True)
os.chdir(WORKDIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "bitsandbytes"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], check=True)
print(f"env_file={ENV_FILE if ENV_FILE.is_file() else 'not present (using runtime/Kaggle secrets)'}")
print("checked_out_source_commit=" + subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
import os

BASE_MODEL = "Qwen/Qwen3-14B"
MODEL_SLUG = "qwen3_14b"
BUCKET_ID = "devaanshpa/CrashDiag"
DATASET_RUN_ID = os.environ.get("CRASHDIAG_DATASET_RUN_ID", "").strip()
def ist_run_id(stage):
    return datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%Y%m%dT%H%M%SIST") + f"-{MODEL_SLUG}-{stage}"
if not DATASET_RUN_ID:
    raise RuntimeError("Set CRASHDIAG_DATASET_RUN_ID to the fresh dataset-generation run ID.")
print(f"base_model={BASE_MODEL}")
print(f"dataset_run_id={DATASET_RUN_ID}")


In [ ]:
from pathlib import Path
from training.artifacts import ArtifactConfig, ArtifactUploader

SFT_RUN_ID = os.environ.get("CRASHDIAG_SFT_RUN_ID", "").strip()
SFT_EVAL_RUN_ID = os.environ.get("CRASHDIAG_SFT_EVAL_RUN_ID") or ist_run_id("sft-eval")
if not SFT_RUN_ID:
    detail = kaggle_secret_errors.get("CRASHDIAG_SFT_RUN_ID") or kaggle_secret_errors.get("SFT_RUN_ID")
    raise RuntimeError("Missing CRASHDIAG_SFT_RUN_ID. Add and enable that Kaggle Secret "
                       "(or SFT_RUN_ID), then rerun from the first cell. "
                       f"Kaggle response: {detail or 'secret was not returned'}")
DATASET_DIR, SFT_DIR = Path("artifacts/datasets"), Path("artifacts/sft")
ArtifactUploader(ArtifactConfig(bucket_id=BUCKET_ID, run_id=DATASET_RUN_ID, token=os.environ["HF_TOKEN"])).download_stage("datasets", DATASET_DIR)
ArtifactUploader(ArtifactConfig(bucket_id=BUCKET_ID, run_id=SFT_RUN_ID, token=os.environ["HF_TOKEN"])).download_stage("sft", SFT_DIR)


In [ ]:
from training.evaluate_jsonl import main as evaluate_main

exit_code = evaluate_main([
    "--model", str(SFT_DIR), "--dataset", str(DATASET_DIR / "grpo_eval.jsonl"),
    "--output-dir", "outputs/sft-eval", "--load-in-4bit", "--precision", "bf16",
    "--max-new-tokens", "64",
    "--sandbox-url", os.environ["CRASHDIAG_SANDBOX_URL"],
    "--artifact-bucket", BUCKET_ID, "--run-id", SFT_EVAL_RUN_ID, "--artifact-stage", "sft-eval",
])
if exit_code: raise RuntimeError(f"SFT evaluation failed: {exit_code}")


In [ ]:
from IPython.display import SVG, display

REPORTS_DIR = Path("outputs/sft-eval") / "reports"
charts = sorted(REPORTS_DIR.glob("*.svg"))
if not charts:
    raise RuntimeError(f"No evaluation SVG charts were generated in {REPORTS_DIR}")
print(f"Uploaded evaluation reports: hf://buckets/{BUCKET_ID}/runs/{SFT_EVAL_RUN_ID}/sft-eval/reports")
for chart in charts:
    display(SVG(filename=str(chart)))
